# 🚦 TrafficPulse: K-Means Clustering ML Analysis for Gurgaon Traffic Flow

**Objective**: Perform multi-factor machine learning analysis on Gurgaon traffic flow telemetry dataset using **K-Means Clustering**, Z-score feature standardization, Elbow Method WCSS evaluation, and cluster fingerprint profiling across 10 multi-dimensional traffic factors.

---

## 1. Import Libraries & Environment Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Styling setup
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
sns.set_palette('crest')
print('Libraries imported successfully.')

## 2. Load Gurgaon Multi-Factor Traffic Dataset
We load `traffic_dataset_gurgaon.csv` containing the **10 Core Traffic Factors**:
1. `Volume` (Flow Rate in veh/hr)
2. `Speed` (Actual speed km/h)
3. `Density` (Vehicles per km)
4. `TimeOfDay` (Hour 00:00 - 23:00)
5. `DayOfWeek` (0 = Sun, 1 = Mon, ..., 6 = Sat)
6. `Seasonality` (Season / Holiday index)
7. `Precipitation` (0: Clear, 1: Rain, 2: Heavy Rain, 3: Snow)
8. `Capacity` (Road design limit veh/hr)
9. `ControlDevices` (0: Highway, 1: Roundabout, 2: Signal, 3: Toll)
10. `VehicleMix` (% Heavy Freight Trucks & Buses)

In [ ]:
# Load dataset
df = pd.read_csv('traffic_dataset_gurgaon.csv')
print(f"Dataset Shape: {df.shape}")
df.head()

## 3. Exploratory Data Analysis (EDA) & Feature Inspection

In [ ]:
# Descriptive statistics
feature_cols = ['Volume', 'Speed', 'Density', 'TimeOfDay', 'DayOfWeek', 'Seasonality', 'Precipitation', 'Capacity', 'ControlDevices', 'VehicleMix']
df[feature_cols].describe()

In [ ]:
# Correlation Matrix Heatmap
plt.figure(figsize=(10, 7))
corr = df[feature_cols].corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Gurgaon Traffic Multi-Factor Feature Correlation Matrix', fontsize=14)
plt.show()

## 4. Feature Standardization (Z-Score Normalization)
Since features have different scales (e.g. `Volume` ~ 3000 vs `Precipitation` 0-3), we apply **Z-Score Normalization** ($Z = \frac{x - \mu}{\sigma}$) using `StandardScaler` to ensure all 10 features contribute equally to Euclidean distance metrics.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[feature_cols])
print("Scaled Matrix Shape:", X_scaled.shape)

## 5. K-Means Algorithm Implementation (From Scratch & Scikit-Learn)
Below is a custom implementation of the **K-Means Clustering Algorithm from Scratch** demonstrating Euclidean distance calculation, centroid initialization, iterative assignment, and convergence.

In [ ]:
class CustomKMeans:
    def __init__(self, k=4, max_iter=50):
        self.k = k
        self.max_iter = max_iter
        self.centroids = None
        self.labels = None
        
    def fit(self, X):
        np.random.seed(42)
        # Randomly choose initial K centroids
        initial_idx = np.random.choice(len(X), self.k, replace=False)
        self.centroids = X[initial_idx]
        
        for iteration in range(self.max_iter):
            # Compute Euclidean distances to all centroids
            distances = np.linalg.norm(X[:, np.newaxis] - self.centroids, axis=2)
            self.labels = np.argmin(distances, axis=1)
            
            # Update centroids
            new_centroids = np.array([X[self.labels == c].mean(axis=0) for c in range(self.k)])
            
            # Check convergence
            if np.allclose(self.centroids, new_centroids):
                break
            self.centroids = new_centroids
        return self

# Run Scratch Model
scratch_kmeans = CustomKMeans(k=4).fit(X_scaled)
print("Custom K-Means fit complete. Centroids Shape:", scratch_kmeans.centroids.shape)

## 6. Elbow Method (WCSS Inertia) & Silhouette Score Evaluation
We evaluate optimal $K$ values between $K=1..6$ to locate the **Elbow Point** (where Within-Cluster Sum of Squares levels off).

In [ ]:
wcss = []
silhouette_scores = []
K_range = range(2, 7)

for k in K_range:
    kmeans_model = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans_model.fit(X_scaled)
    wcss.append(kmeans_model.inertia_)
    score = silhouette_score(X_scaled, kmeans_model.labels_)
    silhouette_scores.append(score)

# Plot Elbow Curve & Silhouette Scores
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(range(2, 7), wcss, marker='o', color='#0284c7', linewidth=2)
ax1.set_title('Elbow Method (WCSS Inertia vs K)', fontsize=12)
ax1.set_xlabel('Number of Clusters (K)')
ax1.set_ylabel('Within-Cluster Sum of Squares (Inertia)')

ax2.plot(range(2, 7), silhouette_scores, marker='s', color='#059669', linewidth=2)
ax2.set_title('Silhouette Score vs K', fontsize=12)
ax2.set_xlabel('Number of Clusters (K)')
ax2.set_ylabel('Silhouette Score')

plt.tight_layout()
plt.show()

## 7. Model Fitting with Optimal K=4 & Cluster Profiling

In [ ]:
# Fit final K-Means model with K=4
final_kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df['ClusterID'] = final_kmeans.fit_predict(X_scaled)

# Map Cluster Labels
cluster_names = {
    0: 'Free Flow (Smooth Traffic)',
    1: 'Moderate Traffic Flow',
    2: 'Heavy Bottleneck',
    3: 'Critical Congestion / Anomaly'
}
df['ClusterName'] = df['ClusterID'].map(cluster_names)

# Display Cluster Distribution & Means
print("Segment Count per Cluster:")
print(df['ClusterName'].value_counts())

df.groupby('ClusterName')[['Volume', 'Speed', 'Density', 'Precipitation', 'VehicleMix']].mean().round(2)

## 8. Cluster Visualization: Volume vs Speed & Density Distributions

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=df, 
    x='Speed', 
    y='Volume', 
    hue='ClusterName', 
    style='ClusterName', 
    s=120, 
    palette=['#059669', '#d97706', '#ea580c', '#dc2626']
)
plt.title('Gurgaon Traffic Clusters: Speed (km/h) vs Volume (veh/hr)', fontsize=14)
plt.xlabel('Average Actual Speed (km/h)')
plt.ylabel('Flow Rate Volume (veh/hr)')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

## 9. Export Processed Cluster Results

In [ ]:
# Export final clustered dataset
output_file = 'traffic_kmeans_results_gurgaon.csv'
df.to_csv(output_file, index=False)
print(f"Successfully exported clustered telemetry results to '{output_file}'.")